# Action Recognition Training - Classification Model

This notebook trains a simple LSTM classifier to recognize walking vs non-walking actions from keypoint sequences.

## 1. Import Libraries

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm

## 2. Configuration

In [ ]:
DATASET_DIR = "../dataset"  # Relative to Behavier_recognition folder
SEQUENCE_LENGTH = 30  # Number of consecutive valid frames
STRIDE = 15  # Sliding window stride
MIN_CONFIDENCE = 0.3  # Minimum keypoint confidence
MIN_VALID_KEYPOINTS = 8  # Minimum valid keypoints per frame (out of 12)
BATCH_SIZE = 64  # Increased from 32 for faster training
NUM_EPOCHS = 50
LEARNING_RATE = 0.001
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.3
K_FOLDS = 5  # Number of folds for K-fold cross-validation
NUM_WORKERS = 4  # Parallel data loading

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():print(f"Batch Size: {BATCH_SIZE} | Num Workers: {NUM_WORKERS}")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"K-Fold CV: {K_FOLDS} folds")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Define Dataset Class

Dataset với bộ lọc chất lượng frame:
- Chỉ sử dụng 12 keypoints quan trọng (vai, tay, hông, chân)
- Lọc frame theo độ tin cậy và số keypoints hợp lệ
- Tự động tìm sequences liên tục chất lượng tốt

In [ ]:
class ActionDataset(Dataset):
    """Dataset for action recognition from keypoints sequences"""
    
    # Lower body keypoints indices (COCO format)
    # 5,6: shoulders, 7,8: elbows, 9,10: wrists, 11,12: hips, 13,14: knees, 15,16: ankles
    BODY_KEYPOINTS = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
    
    def __init__(self, data_paths, labels, sequence_length=30, stride=15, 
                 min_confidence=0.3, min_valid_keypoints=8):
        """
        Args:
            data_paths: List of paths to JSON files
            labels: List of labels (0=non-walking, 1=walking)
            sequence_length: Number of frames per sequence
            stride: Step size for sliding window
            min_confidence: Minimum confidence threshold for keypoints
            min_valid_keypoints: Minimum number of valid keypoints per frame
        """
        self.sequences = []
        self.labels = []
        self.sequence_length = sequence_length
        self.stride = stride
        self.min_confidence = min_confidence
        self.min_valid_keypoints = min_valid_keypoints
        
        print(f"\nLoading dataset with quality filters:")
        print(f"  - Min confidence: {min_confidence}")
        print(f"  - Min valid keypoints: {min_valid_keypoints}/{len(self.BODY_KEYPOINTS)}")
        print(f"  - Sequence length: {sequence_length} frames")
        print(f"  - Stride: {stride} frames")
        
        for path, label in zip(data_paths, labels):
            self._load_json(path, label)
        
        print(f"\nTotal sequences created: {len(self.sequences)}")
    
    def _is_frame_valid(self, frame):
        """Check if frame has enough high-quality keypoints"""
        if not isinstance(frame, list) or len(frame) == 0:
            return False
        
        valid_count = 0
        for kp_idx in self.BODY_KEYPOINTS:
            if kp_idx < len(frame):
                keypoint = frame[kp_idx]
                if len(keypoint) >= 3:
                    confidence = keypoint[2]
                    if confidence >= self.min_confidence:
                        valid_count += 1
        
        return valid_count >= self.min_valid_keypoints
    
    def _find_valid_sequences(self, frames):
        """Find all valid consecutive sequences in frames"""
        valid_sequences = []
        current_sequence = []
        
        for i, frame in enumerate(frames):
            if self._is_frame_valid(frame):
                current_sequence.append(i)
                
                # Check if we have enough consecutive frames
                if len(current_sequence) >= self.sequence_length:
                    # Add sequence with sliding window
                    start_idx = len(current_sequence) - self.sequence_length
                    if start_idx % self.stride == 0 or start_idx == 0:
                        valid_sequences.append(current_sequence[-self.sequence_length:])
            else:
                # Reset if frame is invalid
                current_sequence = []
        
        return valid_sequences
    
    def _load_json(self, path, label):
        """Load JSON and extract valid sequences"""
        try:
            with open(path, 'r') as f:
                data = json.load(f)
            
            frames = data.get('frames', [])
            if len(frames) < self.sequence_length:
                return
            
            # Find all valid consecutive sequences
            valid_sequences = self._find_valid_sequences(frames)
            
            if len(valid_sequences) == 0:
                print(f"⚠️  No valid sequences in {os.path.basename(path)}")
                return
            
            # Extract keypoints from each valid sequence
            for seq_indices in valid_sequences:
                sequence_frames = [frames[i] for i in seq_indices]
                keypoints = self._extract_keypoints(sequence_frames)
                if keypoints is not None:
                    self.sequences.append(keypoints)
                    self.labels.append(label)
            
            if len(valid_sequences) > 0:
                print(f"✓ {os.path.basename(path)}: {len(valid_sequences)} valid sequences")
                
        except Exception as e:
            print(f"❌ Error loading {path}: {e}")
    
    def _extract_keypoints(self, sequence):
        """Extract body keypoints from sequence of frames"""
        keypoints_seq = []
        
        for frame in sequence:
            frame_kps = []
            
            # Extract only body keypoints (arms, legs, hips)
            for kp_idx in self.BODY_KEYPOINTS:
                if kp_idx < len(frame):
                    keypoint = frame[kp_idx]
                    if len(keypoint) >= 3:
                        x, y, conf = keypoint[0], keypoint[1], keypoint[2]
                        
                        # Clip values to valid range
                        x = np.clip(x, 0, 224)
                        y = np.clip(y, 0, 224)
                        conf = np.clip(conf, 0, 1)
                        
                        frame_kps.extend([x, y, conf])
                    else:
                        # Invalid keypoint - use zeros
                        frame_kps.extend([0, 0, 0])
                else:
                    # Missing keypoint - use zeros
                    frame_kps.extend([0, 0, 0])
            
            keypoints_seq.append(frame_kps)
        
        keypoints_array = np.array(keypoints_seq, dtype=np.float32)
        
        # Normalize coordinates to [0, 1]
        keypoints_array[:, 0::3] /= 224.0  # x coordinates
        keypoints_array[:, 1::3] /= 224.0  # y coordinates
        # confidence already in [0, 1]
        
        return keypoints_array
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        sequence = torch.FloatTensor(self.sequences[idx])
        label = torch.LongTensor([self.labels[idx]])[0]
        return sequence, label

## 4. Define Model Architecture

Bi-directional LSTM với fully connected layers

In [ ]:
class ActionClassifier(nn.Module):
    """Bi-directional LSTM classifier for action recognition"""
    
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=2, dropout=0.3):
        super(ActionClassifier, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Bi-directional LSTM
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size * 2, hidden_size)  # *2 for bidirectional
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # x shape: (batch, sequence_length, input_size)
        
        # LSTM forward pass
        lstm_out, _ = self.lstm(x)
        
        # Take the output from the last time step
        lstm_out = lstm_out[:, -1, :]
        
        # Fully connected layers
        out = self.fc1(lstm_out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

In [ ]:
# Load all dataset paths
walking_dir = os.path.join(DATASET_DIR, 'walking')
non_walking_dir = os.path.join(DATASET_DIR, 'non-walking')

data_paths = []
labels = []

# Load walking samples
if os.path.exists(walking_dir):
    for filename in os.listdir(walking_dir):
        if filename.endswith('.json'):
            data_paths.append(os.path.join(walking_dir, filename))
            labels.append(1)  # walking

# Load non-walking samples
if os.path.exists(non_walking_dir):
    for filename in os.listdir(non_walking_dir):
        if filename.endswith('.json'):
            data_paths.append(os.path.join(non_walking_dir, filename))
            labels.append(0)  # non-walking

print(f"Total files: {len(data_paths)}")
print(f"  - Walking: {sum(labels)}")
print(f"  - Non-walking: {len(labels) - sum(labels)}")

data_paths = np.array(data_paths)
labels = np.array(labels)

## 6. K-Fold Cross-Validation Training

Train model với K-fold CV để đánh giá độ chính xác trung bình

In [ ]:
# K-Fold Cross Validation
kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

fold_results = []
all_histories = []

print("\n" + "="*60)
print(f"K-FOLD CROSS-VALIDATION ({K_FOLDS} folds)")
print("="*60)

for fold, (train_idx, val_idx) in enumerate(kfold.split(data_paths)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{K_FOLDS}")
    print(f"{'='*60}")
    
    # Split data for this fold
    train_paths_fold = data_paths[train_idx]
    train_labels_fold = labels[train_idx]
    val_paths_fold = data_paths[val_idx]
    val_labels_fold = labels[val_idx]
    
    print(f"\nFold {fold + 1} split:")
    print(f"  Train files: {len(train_paths_fold)}")
    print(f"  Val files: {len(val_paths_fold)}")
    
    # Create datasets
    print("\nCreating train dataset...")
    train_dataset = ActionDataset(
        train_paths_fold, train_labels_fold,
        SEQUENCE_LENGTH, STRIDE,
        MIN_CONFIDENCE, MIN_VALID_KEYPOINTS
    )
    
    print("\nCreating validation dataset...")
    val_dataset = ActionDataset(
        val_paths_fold, val_labels_fold,
        SEQUENCE_LENGTH, STRIDE,
        MIN_CONFIDENCE, MIN_VALID_KEYPOINTS
    )
    
    if len(train_dataset) == 0 or len(val_dataset) == 0:
        print(f"⚠️ Fold {fold + 1}: No valid sequences, skipping...")
        continue
    
    print(f"\nDataset sequences:")
    print(f"  Train: {len(train_dataset)}")
    print(f"  Val: {len(val_dataset)}")
    
    # Create dataloaders with optimization
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=NUM_WORKERS,
        pin_memory=True if torch.cuda.is_available() else False,
        persistent_workers=True if NUM_WORKERS > 0 else False
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=BATCH_SIZE * 2,  # Larger batch for validation (no gradients)
        shuffle=False, 
        num_workers=NUM_WORKERS,
        pin_memory=True if torch.cuda.is_available() else False,
        persistent_workers=True if NUM_WORKERS > 0 else False
    )
    
    # Get input size
    sample_seq, _ = train_dataset[0]
    input_size = sample_seq.shape[1]
    
    # Create new model for this fold
    model = ActionClassifier(
        input_size=input_size,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        num_classes=2,
        dropout=DROPOUT
    ).to(device)
    
    # Training setup
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    
    # Training history
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    best_val_acc = 0.0
    
    # Training loop
    for epoch in range(NUM_EPOCHS):
        # Training phase
        model.train()
        train_loss = 0.0
        train_preds = []
        train_labels_list = []
        
        for sequences, batch_labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
            sequences = sequences.to(device)
            batch_labels = batch_labels.to(device)
            
            optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_preds.extend(predicted.cpu().numpy())
            train_labels_list.extend(batch_labels.cpu().numpy())
        
        train_loss /= len(train_loader)
        train_acc = accuracy_score(train_labels_list, train_preds)
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_preds = []
        val_labels_list = []
        
        with torch.no_grad():
            for sequences, batch_labels in val_loader:
                sequences = sequences.to(device)
                batch_labels = batch_labels.to(device)
                
                outputs = model(sequences)
                loss = criterion(outputs, batch_labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_preds.extend(predicted.cpu().numpy())
                val_labels_list.extend(batch_labels.cpu().numpy())
        
        val_loss /= len(val_loader)
        val_acc = accuracy_score(val_labels_list, val_preds)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        scheduler.step(val_acc)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")
        
        # Save best model for this fold
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_acc': best_val_acc,
                'train_losses': train_losses,
                'val_losses': val_losses,
                'train_accs': train_accs,
                'val_accs': val_accs
            }
            torch.save(checkpoint, f'action_classifier_fold{fold+1}.pth')
            if (epoch + 1) % 5 == 0:
                print(f"  ✓ Saved best model (Val Acc: {best_val_acc:.4f})")
    
    # Store fold results
    fold_results.append({
        'fold': fold + 1,
        'best_val_acc': best_val_acc,
        'final_train_acc': train_accs[-1],
        'final_val_acc': val_accs[-1],
        'val_preds': val_preds,
        'val_labels': val_labels_list
    })
    
    all_histories.append({
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs
    })
    
    print(f"\nFold {fold + 1} Results:")
    print(f"  Best Val Accuracy: {best_val_acc:.4f}")
    print(f"  Final Train Accuracy: {train_accs[-1]:.4f}")
    print(f"  Final Val Accuracy: {val_accs[-1]:.4f}")
    
    # Print classification report for this fold
    print(f"\nFold {fold + 1} Classification Report:")
    print(classification_report(val_labels_list, val_preds, 
                                target_names=['Non-Walking', 'Walking'],
                                zero_division=0))

print("\n" + "="*60)
print("K-FOLD CROSS-VALIDATION COMPLETE")
print("="*60)

Epoch 41/50: 100%|██████████| 535/535 [00:02<00:00, 253.67it/s]

Epoch 42/50: 100%|██████████| 535/535 [00:02<00:00, 264.19it/s]

Epoch 43/50: 100%|██████████| 535/535 [00:02<00:00, 264.34it/s]

Epoch 44/50: 100%|██████████| 535/535 [00:02<00:00, 231.49it/s]

Epoch 45/50: 100%|██████████| 535/535 [00:06<00:00, 83.19it/s] 

Epoch 46/50:   4%|▍         | 22/535 [00:00<00:02, 214.46it/s]

## 5. Load and Prepare Dataset

Load files và tạo train/val split

In [ ]:
# Calculate average metrics across all folds
avg_best_val_acc = np.mean([r['best_val_acc'] for r in fold_results])
avg_final_train_acc = np.mean([r['final_train_acc'] for r in fold_results])
avg_final_val_acc = np.mean([r['final_val_acc'] for r in fold_results])

std_best_val_acc = np.std([r['best_val_acc'] for r in fold_results])
std_final_val_acc = np.std([r['final_val_acc'] for r in fold_results])

print("\n" + "="*60)
print("AVERAGE RESULTS ACROSS ALL FOLDS")
print("="*60)
print(f"\nBest Validation Accuracy:")
print(f"  Mean: {avg_best_val_acc:.4f} ± {std_best_val_acc:.4f}")
print(f"\nFinal Training Accuracy:")
print(f"  Mean: {avg_final_train_acc:.4f}")
print(f"\nFinal Validation Accuracy:")
print(f"  Mean: {avg_final_val_acc:.4f} ± {std_final_val_acc:.4f}")

print(f"\n\nPer-fold results:")
for r in fold_results:
    print(f"  Fold {r['fold']}: Best Val Acc = {r['best_val_acc']:.4f}")

In [ ]:
# Visualize K-fold results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Per-fold best validation accuracy
fold_nums = [r['fold'] for r in fold_results]
best_accs = [r['best_val_acc'] for r in fold_results]

axes[0, 0].bar(fold_nums, best_accs, color='steelblue', alpha=0.7)
axes[0, 0].axhline(y=avg_best_val_acc, color='red', linestyle='--', label=f'Mean: {avg_best_val_acc:.4f}')
axes[0, 0].set_xlabel('Fold')
axes[0, 0].set_ylabel('Best Validation Accuracy')
axes[0, 0].set_title('Best Validation Accuracy per Fold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Training curves for all folds
for i, h in enumerate(all_histories):
    axes[0, 1].plot(h['train_losses'], alpha=0.5, label=f"Fold {i+1}")
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Training Loss')
axes[0, 1].set_title('Training Loss Curves (All Folds)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Validation accuracy curves
for i, h in enumerate(all_histories):
    axes[1, 0].plot(h['val_accs'], alpha=0.5, label=f"Fold {i+1}")
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Validation Accuracy')
axes[1, 0].set_title('Validation Accuracy Curves (All Folds)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Accuracy distribution
axes[1, 1].boxplot([best_accs], labels=['Best Val Acc'])
axes[1, 1].axhline(y=avg_best_val_acc, color='red', linestyle='--', alpha=0.7)
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title('Accuracy Distribution Across Folds')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('kfold_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved to 'kfold_results.png'")

In [ ]:
# Aggregate confusion matrix across all folds
from sklearn.metrics import confusion_matrix

# Collect all predictions and true labels
all_true = []
all_pred = []

for r in fold_results:
    all_true.extend(r['val_labels'])
    all_pred.extend(r['val_preds'])

# Calculate overall confusion matrix
cm = confusion_matrix(all_true, all_pred)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

# Set labels
class_names = ['Non-walking', 'Walking']
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names, yticklabels=class_names,
       title='Aggregated Confusion Matrix (All Folds)',
       ylabel='True label',
       xlabel='Predicted label')

# Rotate the tick labels
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Add text annotations
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=16)

plt.tight_layout()
plt.savefig('confusion_matrix_kfold.png', dpi=150, bbox_inches='tight')
plt.show()

# Print overall metrics
from sklearn.metrics import classification_report
print("\nOverall Classification Report (All Folds Aggregated):")
print(classification_report(all_true, all_pred, target_names=class_names, digits=4))

print("\n" + "="*60)
print("K-FOLD CROSS-VALIDATION COMPLETE!")
print("="*60)
print(f"\nFinal Result: {avg_best_val_acc:.4f} ± {std_best_val_acc:.4f}")
print(f"\nBest performing fold: Fold {fold_results[np.argmax(best_accs)]['fold']}")
print(f"  Accuracy: {max(best_accs):.4f}")
print(f"\nModels saved:")
for i in range(K_FOLDS):
    print(f"  - action_classifier_fold{i+1}.pth")

In [ ]:
# Load dataset paths
walking_dir = os.path.join(DATASET_DIR, 'walking')
non_walking_dir = os.path.join(DATASET_DIR, 'non-walking')

data_paths = []
labels = []

# Load walking samples
if os.path.exists(walking_dir):
    for filename in os.listdir(walking_dir):
        if filename.endswith('.json'):
            data_paths.append(os.path.join(walking_dir, filename))
            labels.append(1)  # walking

# Load non-walking samples
if os.path.exists(non_walking_dir):
    for filename in os.listdir(non_walking_dir):
        if filename.endswith('.json'):
            data_paths.append(os.path.join(non_walking_dir, filename))
            labels.append(0)  # non-walking

print(f"Found {sum(labels)} walking and {len(labels) - sum(labels)} non-walking files")

# Split dataset
train_paths, val_paths, train_labels, val_labels = train_test_split(
    data_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"\nDataset split:")
print(f"  Train files: {len(train_paths)}")
print(f"  Val files: {len(val_paths)}")

In [ ]:
# Create datasets with quality filtering
print("\n" + "="*60)
print("CREATING TRAIN DATASET")
print("="*60)
train_dataset = ActionDataset(
    train_paths, train_labels, 
    SEQUENCE_LENGTH, STRIDE,
    MIN_CONFIDENCE, MIN_VALID_KEYPOINTS
)

print("\n" + "="*60)
print("CREATING VALIDATION DATASET")
print("="*60)
val_dataset = ActionDataset(
    val_paths, val_labels,
    SEQUENCE_LENGTH, STRIDE,
    MIN_CONFIDENCE, MIN_VALID_KEYPOINTS
)

print(f"\nFinal dataset:")
print(f"  Train sequences: {len(train_dataset)}")
print(f"  Val sequences: {len(val_dataset)}")